In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

class ROCIndicator:
    def __init__(self, close, window, fillna=False): self.close = close
    def roc(self): return [0.0 for _ in range(len(self.close))]

class RSIIndicator:
    def __init__(self, close, window, fillna=False): self.close = close
    def rsi(self): return [50.0 for _ in range(len(self.close))]

class StochasticOscillator:
    def __init__(self, close, high, low, window, smooth_window, fillna=False): self.close = close
    def stoch(self): return pl.Series("stoch", [20.0 for _ in range(len(self.close))])
    def stoch_signal(self): return pl.Series("stoch_signal", [25.0 for _ in range(len(self.close))])

class StochRSIIndicator:
    def __init__(self, close, window, smooth1, smooth2, fillna=False): self.close = close
    def stochrsi(self): return [0.1 for _ in range(len(self.close))]
    def stochrsi_d(self): return [0.2 for _ in range(len(self.close))]
    def stochrsi_k(self): return [0.3 for _ in range(len(self.close))]

fillna = True  # passed to ta indicators

# Register the existing indicator doubles under the real import path so
# generated code can retain the source project's external dependency import.
import sys
import types
_ta_module = types.ModuleType("ta")
_ta_momentum_module = types.ModuleType("ta.momentum")
_ta_momentum_module.ROCIndicator = ROCIndicator
_ta_momentum_module.RSIIndicator = RSIIndicator
_ta_momentum_module.StochasticOscillator = StochasticOscillator
_ta_momentum_module.StochRSIIndicator = StochRSIIndicator
_ta_module.momentum = _ta_momentum_module
sys.modules["ta"] = _ta_module
sys.modules["ta.momentum"] = _ta_momentum_module


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- portfolio shared ---
PORTFOLIO_BASE_PD = pd.DataFrame({"Close": [100.0, 101.0, 102.0, 103.0, 104.0], "High": [101.0, 102.0, 103.0, 104.0, 105.0], "Low": [99.0, 100.0, 101.0, 102.0, 103.0]})
PORTFOLIO_BASE_PL = pl.from_pandas(PORTFOLIO_BASE_PD)

def _set_portfolio_self_pd():
    global self
    self = SimpleNamespace(df=PORTFOLIO_BASE_PD.copy())

def _set_portfolio_self_pl():
    global self
    self = SimpleNamespace(df=PORTFOLIO_BASE_PL.clone())

# --- portfolio_momentum_roc ---
FIX_PORTFOLIO_MOMENTUM_ROC_WINDOW = 14

# --- portfolio_momentum_rsi ---
FIX_PORTFOLIO_MOMENTUM_RSI_WINDOW = 14

# --- portfolio_momentum_stoch_osc ---
FIX_PORTFOLIO_MOMENTUM_STOCH_OSC_SMOOTH_WINDOW = 14
FIX_PORTFOLIO_MOMENTUM_STOCH_OSC_WINDOW = 14

# --- portfolio_momentum_stoch_rsi ---
FIX_PORTFOLIO_MOMENTUM_STOCH_RSI_SMOOTH1 = 14
FIX_PORTFOLIO_MOMENTUM_STOCH_RSI_SMOOTH2 = 14
FIX_PORTFOLIO_MOMENTUM_STOCH_RSI_WINDOW = 14

_set_portfolio_self_pd()
print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_portfolio_momentum_roc(window):
    self.df['ROC'] = ROCIndicator(close=self.df.Close, window=window, fillna=fillna).roc()
    return None

def before_portfolio_momentum_rsi(window):
    self.df['RSI'] = RSIIndicator(close=self.df.Close, window=window, fillna=fillna).rsi()
    return None

def before_portfolio_momentum_stoch_osc(smooth_window, window):
    stochastic_oscillator = StochasticOscillator(
        close=self.df.Close, high=self.df.High, low=self.df.Low, window=window, smooth_window=smooth_window,
        fillna=fillna
    )
    self.df['stoch'] = stochastic_oscillator.stoch()
    self.df['stoch_signal'] = stochastic_oscillator.stoch_signal()
    return stochastic_oscillator

def before_portfolio_momentum_stoch_rsi(smooth1, smooth2, window):
    stoch_rsi = StochRSIIndicator(
        close=self.df.Close, window=window, smooth1=smooth1, smooth2=smooth2, fillna=fillna
    )
    self.df['stoch_rsi'] = stoch_rsi.stochrsi()
    self.df['stoch_rsi_d'] = stoch_rsi.stochrsi_d()
    self.df['stoch_rsi_k'] = stoch_rsi.stochrsi_k()
    return stoch_rsi

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_portfolio_momentum_roc(window):
    self.df = self.df.with_columns(
        pd.Series(ROCIndicator(close=self.df["Close"], window=window, fillna=fillna).roc()).alias("ROC")
    )
    return None

def gen_portfolio_momentum_rsi(window):

    self.df = self.df.with_columns(
        pl.when(
            pl.when(pl.col("Close").diff() < 0)
            .then(-pl.col("Close").diff())
            .otherwise(0.0)
            .ewm_mean(alpha=1 / window, adjust=False)
            == 0
        )
        .then(100.0)
        .otherwise(
            100.0
            - (
                100.0
                / (
                    1.0
                    + (
                        pl.when(pl.col("Close").diff() > 0)
                        .then(pl.col("Close").diff())
                        .otherwise(0.0)
                        .ewm_mean(alpha=1 / window, adjust=False)
                        / pl.when(pl.col("Close").diff() < 0)
                        .then(-pl.col("Close").diff())
                        .otherwise(0.0)
                        .ewm_mean(alpha=1 / window, adjust=False)
                    )
                )
            )
        )
        .alias("RSI")
    )
    return None

def gen_portfolio_momentum_stoch_osc(smooth_window, window):
    pd = pl  # LLM used `import polars as pd`
    stochastic_oscillator = StochasticOscillator(
        close=self.df["Close"], high=self.df["High"], low=self.df["Low"], window=window, smooth_window=smooth_window,
        fillna=fillna
    )
    self.df = self.df.with_columns([
        stochastic_oscillator.stoch().alias("stoch"),
        stochastic_oscillator.stoch_signal().alias("stoch_signal")
    ])
    return stochastic_oscillator

def gen_portfolio_momentum_stoch_rsi(smooth1, smooth2, window):
    stoch_rsi = StochRSIIndicator(
        close=self.df["Close"], window=window, smooth1=smooth1, smooth2=smooth2, fillna=fillna
    )
    self.df = self.df.with_columns([
        pl.Series("stoch_rsi", stoch_rsi.stochrsi()),
        pl.Series("stoch_rsi_d", stoch_rsi.stochrsi_d()),
        pl.Series("stoch_rsi_k", stoch_rsi.stochrsi_k()),
    ])
    return stoch_rsi

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def _comparison_label(label):
    text = str(label)
    if text.lstrip().startswith(("L2", "L3")):
        return text
    return f"L2 equivalence {text}"

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: portfolio_momentum_rsi ===

# L1 smoke – generated
try:
    _set_portfolio_self_pl()
    _r = gen_portfolio_momentum_rsi(FIX_PORTFOLIO_MOMENTUM_RSI_WINDOW)
    print("✅ L1 smoke gen_portfolio_momentum_rsi: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_portfolio_momentum_rsi: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _set_portfolio_self_pd()
    _rb = before_portfolio_momentum_rsi(FIX_PORTFOLIO_MOMENTUM_RSI_WINDOW)
    print("✅ L1 smoke before_portfolio_momentum_rsi: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_portfolio_momentum_rsi: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _set_portfolio_self_pd()
    before_portfolio_momentum_rsi(FIX_PORTFOLIO_MOMENTUM_RSI_WINDOW)
    _before_df = self.df.copy()
    _set_portfolio_self_pl()
    gen_portfolio_momentum_rsi(FIX_PORTFOLIO_MOMENTUM_RSI_WINDOW)
    _gen_df = self.df.clone()
    compare(_before_df, _gen_df, "portfolio_momentum_rsi")
except Exception as _e:
    print(f"❌ L2 equivalence portfolio_momentum_rsi: setup error — {type(_e).__name__}: {_e}")

# L3 edge – empty price history; compare the mutated object state on both sides.
try:
    _empty_pd = pd.DataFrame({"Close": pd.Series(dtype=float), "High": pd.Series(dtype=float), "Low": pd.Series(dtype=float)})
    _empty_pl = pl.DataFrame(schema={"Close": pl.Float64, "High": pl.Float64, "Low": pl.Float64})
    self = SimpleNamespace(df=_empty_pd)
    before_portfolio_momentum_rsi(FIX_PORTFOLIO_MOMENTUM_RSI_WINDOW)
    _before_empty = self.df.copy()
    self = SimpleNamespace(df=_empty_pl)
    gen_portfolio_momentum_rsi(FIX_PORTFOLIO_MOMENTUM_RSI_WINDOW)
    _gen_empty = self.df.clone()
    compare(_before_empty, _gen_empty, "L3 edge portfolio_momentum_rsi empty frame", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge portfolio_momentum_rsi: {type(_e).__name__}: {_e}")
